# PyTorch CNN Wind Anomaly Detection

This notebook adds a CPU-friendly PyTorch anomaly-detection workflow based on compact 1D convolutional autoencoders. It focuses on `wind_speed_10m` while still using a small multivariate weather slice so anomaly scores have more context than a single raw series.

The workflow is:

- build rolling hourly windows from the canonical Open-Meteo table
- train compact 1D CNN autoencoders on chronological training windows
- compare multiple CPU-friendly architectures side by side
- rank unusual windows by reconstruction error and inspect recent anomalies

In [ ]:
from __future__ import annotations

import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_meteo import DEFAULT_DATA_DIR, build_hourly_table

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device("cpu")
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print({"device": str(device), "torch_version": torch.__version__})

## Load the canonical hourly table

The notebook reuses the shared project loader and keeps a compact multivariate feature slice centered on wind behavior.

In [ ]:
table = build_hourly_table(DEFAULT_DATA_DIR)
column_arrays = {
    name: np.asarray(values, dtype=float) for name, values in table.columns.items()
}
timestamps = np.asarray(table.timestamps)

primary_name = "wind__wind_speed_10m"
feature_names = [
    "wind__wind_speed_10m",
    "wind__wind_gusts_10m",
    "misc__pressure_msl",
    "misc__precipitation",
    "misc__temperature_2m",
    "cloud_cover__cloud_cover",
    "solar_radiation_vars__shortwave_radiation",
    "additional_vars_and_options__is_day",
    "european_air_quality_index__european_aqi",
]
missing = [name for name in feature_names if name not in column_arrays]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

feature_matrix = np.column_stack(
    [column_arrays[name] for name in feature_names]
).astype(np.float32)

print(f"Rows: {len(timestamps):,}")
print(f"Feature columns: {len(feature_names)}")
print(f"Start: {timestamps[0]}")
print(f"End:   {timestamps[-1]}")

## Build rolling windows for anomaly detection

Each sample is a 24-hour multivariate window. The benchmark uses a recent bounded subset so the autoencoder comparison stays practical on CPU.

In [ ]:
window_size = 24
max_windows = 16000

windows = []
window_times = []
window_primary = []

for end_index in range(window_size, len(timestamps) + 1):
    window = feature_matrix[end_index - window_size : end_index]
    if np.isfinite(window).all():
        windows.append(window)
        window_times.append(timestamps[end_index - 1])
        window_primary.append(window[:, 0])

X = np.asarray(windows, dtype=np.float32)
time_index = np.asarray(window_times)
primary_windows = np.asarray(window_primary, dtype=np.float32)

if len(X) > max_windows:
    X = X[-max_windows:]
    time_index = time_index[-max_windows:]
    primary_windows = primary_windows[-max_windows:]

print(f"Windows: {len(X):,}")
print(f"Window shape: {X.shape[1:]}")
print(f"Subset start: {time_index[0]}")
print(f"Subset end:   {time_index[-1]}")

In [ ]:
def chronological_split(
    X: np.ndarray,
    time_index: np.ndarray,
    primary_windows: np.ndarray,
    train_ratio: float = 0.7,
    valid_ratio: float = 0.15,
):
    total = len(X)
    train_end = int(total * train_ratio)
    valid_end = int(total * (train_ratio + valid_ratio))
    return {
        "X_train": X[:train_end],
        "t_train": time_index[:train_end],
        "p_train": primary_windows[:train_end],
        "X_valid": X[train_end:valid_end],
        "t_valid": time_index[train_end:valid_end],
        "p_valid": primary_windows[train_end:valid_end],
        "X_test": X[valid_end:],
        "t_test": time_index[valid_end:],
        "p_test": primary_windows[valid_end:],
    }


split = chronological_split(X, time_index, primary_windows)
train_flat = split["X_train"].reshape(-1, split["X_train"].shape[-1])
x_mean = train_flat.mean(axis=0)
x_std = train_flat.std(axis=0) + 1e-6


def normalize_features(values: np.ndarray) -> np.ndarray:
    return ((values - x_mean) / x_std).astype(np.float32)


def to_tensor_dataset(X_values: np.ndarray) -> TensorDataset:
    normalized = torch.from_numpy(normalize_features(X_values)).permute(0, 2, 1)
    return TensorDataset(normalized)


train_dataset = to_tensor_dataset(split["X_train"])
valid_dataset = to_tensor_dataset(split["X_valid"])
test_dataset = to_tensor_dataset(split["X_test"])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

for name in ["train", "valid", "test"]:
    print(name, split[f"X_{name}"].shape, split[f"t_{name}"][0], split[f"t_{name}"][-1])

## Define compact 1D CNN autoencoders

These models reconstruct multivariate hourly windows. Reconstruction error becomes the anomaly score, and the benchmark compares several lightweight CNN variants side by side.

In [ ]:
class CNNAutoencoder(nn.Module):
    def __init__(
        self,
        input_channels: int,
        latent_channels: int,
        hidden_channels: int,
        kernel_size: int,
    ):
        super().__init__()
        padding = kernel_size // 2
        self.encoder = nn.Sequential(
            nn.Conv1d(
                input_channels,
                hidden_channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(),
            nn.Conv1d(
                hidden_channels,
                latent_channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv1d(
                latent_channels,
                hidden_channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
            nn.ReLU(),
            nn.Conv1d(
                hidden_channels,
                input_channels,
                kernel_size=kernel_size,
                padding=padding,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        encoded = self.encoder(x)
        return self.decoder(encoded)


def reconstruction_errors(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    errors = []
    with torch.no_grad():
        for (features,) in loader:
            features = features.to(device)
            recon = model(features)
            batch_errors = torch.mean((recon - features) ** 2, dim=(1, 2))
            errors.append(batch_errors.cpu().numpy())
    return np.concatenate(errors)


def train_autoencoder(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    epochs: int = 10,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    history = []
    best_valid_loss = float("inf")
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for (features,) in train_loader:
            features = features.to(device)
            optimizer.zero_grad()
            recon = model(features)
            loss = loss_fn(recon, features)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = float(np.mean(batch_losses))
        valid_errors = reconstruction_errors(model, valid_loader)
        valid_loss = float(np.mean(valid_errors))
        history.append(
            {"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss}
        )

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }

    if best_state is not None:
        model.load_state_dict(best_state)

    return history

## Compare CNN anomaly detectors

Each model is trained on the same chronological training windows. Validation errors define a high-error threshold, and test windows above that threshold are treated as anomalies for ranking and inspection.

In [ ]:
benchmark_specs = {
    "cnn_autoencoder_small": {
        "latent_channels": 8,
        "hidden_channels": 16,
        "kernel_size": 3,
    },
    "cnn_autoencoder_medium": {
        "latent_channels": 12,
        "hidden_channels": 24,
        "kernel_size": 3,
    },
    "cnn_autoencoder_wide_kernel": {
        "latent_channels": 12,
        "hidden_channels": 24,
        "kernel_size": 5,
    },
}

benchmark_rows = []
histories = {}
anomaly_results = {}

for model_name, spec in benchmark_specs.items():
    model = CNNAutoencoder(
        input_channels=len(feature_names),
        latent_channels=spec["latent_channels"],
        hidden_channels=spec["hidden_channels"],
        kernel_size=spec["kernel_size"],
    ).to(device)
    history = train_autoencoder(
        model, train_loader, valid_loader, epochs=10, lr=1e-3, weight_decay=1e-4
    )
    train_errors = reconstruction_errors(model, train_loader)
    valid_errors = reconstruction_errors(model, valid_loader)
    test_errors = reconstruction_errors(model, test_loader)
    threshold = float(np.quantile(valid_errors, 0.99))
    anomaly_mask = test_errors >= threshold
    top_order = np.argsort(test_errors)[-10:][::-1]
    benchmark_rows.append(
        {
            "model": model_name,
            "train_error_mean": float(np.mean(train_errors)),
            "valid_error_mean": float(np.mean(valid_errors)),
            "test_error_mean": float(np.mean(test_errors)),
            "threshold": threshold,
            "flagged_windows": int(anomaly_mask.sum()),
        }
    )
    histories[model_name] = history
    anomaly_results[model_name] = {
        "test_errors": test_errors,
        "threshold": threshold,
        "anomaly_mask": anomaly_mask,
        "top_order": top_order,
    }

for row in sorted(benchmark_rows, key=lambda item: item["test_error_mean"]):
    print(row)

In [ ]:
best_model_name = min(benchmark_rows, key=lambda item: item["test_error_mean"])["model"]
best_result = anomaly_results[best_model_name]

fig, axes = plt.subplots(
    1, len(histories), figsize=(5 * len(histories), 4), sharey=True
)
axes = np.atleast_1d(axes)
for ax, (model_name, history) in zip(axes, histories.items()):
    epochs = [row["epoch"] for row in history]
    train_loss = [row["train_loss"] for row in history]
    valid_loss = [row["valid_loss"] for row in history]
    ax.plot(epochs, train_loss, label="train")
    ax.plot(epochs, valid_loss, label="valid")
    ax.set_title(model_name)
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set_ylabel("Reconstruction MSE")
plt.tight_layout()

plot_count = min(24 * 10, len(split["p_test"]))
recent_primary = split["p_test"][-plot_count:, -1]
recent_times = split["t_test"][-plot_count:]
recent_errors = best_result["test_errors"][-plot_count:]
recent_mask = best_result["anomaly_mask"][-plot_count:]

fig, ax1 = plt.subplots()
ax1.plot(recent_times, recent_primary, label="wind_speed_10m", linewidth=2)
ax1.scatter(
    recent_times[recent_mask],
    recent_primary[recent_mask],
    color="crimson",
    label="flagged anomalies",
    zorder=3,
)
ax1.set_xlabel("Timestamp")
ax1.set_ylabel("Wind speed 10m")
ax1.set_title(f"Recent wind anomalies flagged by {best_model_name}")
ax2 = ax1.twinx()
ax2.plot(
    recent_times,
    recent_errors,
    color="darkorange",
    alpha=0.55,
    label="reconstruction error",
)
ax2.axhline(best_result["threshold"], color="gray", linestyle="--", label="threshold")
ax2.set_ylabel("Reconstruction error")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.xticks(rotation=45)
plt.tight_layout()

print("Top anomaly windows for best model:")
for index in best_result["top_order"]:
    print(split["t_test"][index], float(best_result["test_errors"][index]))

## Suggested follow-up experiments

- widen the window from 24 hours to 48 or 72 hours
- add derived wind features such as gust-to-speed ratio and local differences
- compare CNN anomaly scores with the sklearn anomaly detectors on the same sample window
- switch the primary variable to `temperature_2m` or `european_aqi` while keeping the same CNN autoencoder structure